# AgentMeter — Pilot (Google Colab, free T4)

Per-agent resource benchmarking of one open-source LLM in a **minimal linear**
Perceive → Reason → Decide → Act network-threat-detection pipeline.

Dataset: `data/cicids_pilot.csv` (real CIC-IDS flows, 78 features). Taxonomy is
**5 classes**: Brute Force, Volumetric DDoS, Port Scanning, DoS Hulk, Benign.

**Tahap 1 guardrails (unchanged):**
- Pipeline is the *test subject* only — minimal, generic, linear. No governance,
  correlation, alerting, or retry/self-correction loops.
- **Sequential** execution only (never parallel — it contaminates readings).
- **Mode A**: each model is pulled from the HF Hub and run in-process on the GPU,
  so per-agent VRAM is measurable via `torch.cuda` / `pynvml`.
- **Subprocess-per-model VRAM isolation**: each model runs in its **own worker
  subprocess** (`agentmeter.worker`) that loads exactly one model, measures its
  fresh-context baseline, runs the scenarios, then **exits** — so the OS reclaims
  all GPU memory and the next model starts from a clean CUDA context. This
  reliably frees bitsandbytes 4-bit / `device_map` weights that in-process unload
  could not. Each model's `vram_guard` verifies it *started* clean.
- All config in `config.yaml` / the pilot config. Nothing hard-coded.
- GPU required: if `torch.cuda` is unavailable the pilot **STOPS** (no CPU fallback).

### ⚠️ Quantization notice (declare in your thesis)
A 7–8B model does **not** fit in fp16 on a 15 GB T4. This pilot uses **4-bit NF4**
quantization (bitsandbytes). The VRAM / latency / token numbers are therefore for
the **quantized** model, and any model-vs-model comparison must use the **same**
quantization to stay fair. This is a methodological choice you must state.

### Cost
Colab free tier has no \$ cost, but sessions are time-limited and the GPU can be
reclaimed. Save the result JSON files as soon as the run finishes.

## 1. Confirm you have a GPU runtime
Runtime → Change runtime type → Hardware accelerator = **T4 GPU**. Then run:

In [ ]:
!nvidia-smi -L || echo 'NO GPU: set Runtime -> Change runtime type -> T4 GPU'

## 2. Get the AgentMeter code
Clones the repo and checks out the working branch. If the repo is **private**,
paste a GitHub token when prompted (input is hidden). If it is public, just press
Enter to skip.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/cool-ride-mitmzl'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('AgentMeter'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, url + '.git', 'AgentMeter'], check=True)
os.chdir('AgentMeter')
subprocess.run(['git', 'checkout', BRANCH], check=True)
print('cwd:', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())

## 3. Install dependencies
Colab already ships a CUDA build of `torch` — we do **not** reinstall it (that
would risk breaking CUDA). We add the CPU-set deps plus the GPU/HF extras and
`bitsandbytes` for 4-bit.

In [ ]:
# CPU-set deps (langgraph, pandas, numpy, scipy, pyyaml) — no torch here
!pip install -q -r requirements.txt
# GPU / HF extras (torch already present on Colab)
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — switch runtime to T4 before continuing.'

## 4. Authenticate with your Hugging Face token (secure)
Entered via `getpass` — **not** hardcoded, not printed, not saved to the notebook.
You must have accepted the model's gated licence on its Hub page first.

Alternatively use a Colab Secret named `HF_TOKEN` (🔑 panel) — the cell picks it up.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 5. Choose the models to benchmark
Both models run **sequentially** (never in parallel) at the same **4-bit NF4**
quantization, same config, same scenarios — so the comparison is fair. Accept
each model's gated licence on its HF page first, or the download will 401.

In [ ]:
CONFIG = 'configs/pilot_colab_t4.yaml'
MODELS = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    'meta-llama/Meta-Llama-3-8B-Instruct',
]
print('Will benchmark, one after another:')
for m in MODELS: print('  -', m)

## 6. Run the models (sequential, subprocess-per-model VRAM isolation)
For **each** model, in its own fresh **worker subprocess**: measure the
fresh-context baseline → load (4-bit) → run all scenarios → write
`results/pilot_<model>.json` → **the process exits and the OS reclaims all VRAM**
→ next model. Because every worker is a brand-new process, each model starts from
a clean CUDA context (this is what finally frees the bitsandbytes 4-bit /
`device_map` weights). Never two at once. Also writes `results/pilot_combined.json`.
First load of each downloads its weights (~4–5 GB in 4-bit; one-time, excluded
from per-agent timings).

> **Tip:** for a quick isolation check first, set `n=1` below (2 models, ~1 min).
> `device_vram_before_load` should be near-identical for both models and every
> `vram_guard.exceeded` should be `False`. Then re-run with `n=10` for the pilot.

In [ ]:
from agentmeter.pilot import run_pilot_models

# n=1 for a quick VRAM-isolation check; n=10 for the actual pilot.
results = run_pilot_models(models=MODELS, config_path=CONFIG, n=10)

## 7. Verify VRAM isolation (the key check)
Confirms the subprocess isolation worked. Every model should have started in a
clean fresh context: `device_vram_before_load` near the shared baseline and
`vram_guard.exceeded == False`. `in_process_residual_mb` is a **harmless**
diagnostic — the worker process exits and the OS reclaims that VRAM, so it does
**not** mean the run was dirty.

In [ ]:
import json
c = json.load(open('results/pilot_combined.json'))
print('fresh-context baseline (MB):', c.get('baseline_vram_mb'))
print()
print(f"{'model':<40} {'before_load':>11} {'guard.exceeded':>15} {'in_proc_resid':>14}")
print('-' * 84)
for m in c['models']:
    g = m.get('vram_guard') or {}
    print(f"{str(m.get('model_label')):<40} {str(m.get('device_vram_before_load_mb')):>11} "
          f"{str(g.get('exceeded')):>15} {str(m.get('in_process_residual_mb')):>14}")
clean = all(not (m.get('vram_guard') or {}).get('exceeded') for m in c['models'])
print()
print('ISOLATION:', 'OK — every model started clean.' if clean
      else 'PROBLEM — a model started contaminated (see the WARNING above).')

## 8. Download the result files
Send these back for the side-by-side comparison demo: the two per-model
files and the combined file.

In [ ]:
import glob
from google.colab import files
for f in sorted(glob.glob('results/pilot_*.json')):
    print('downloading', f)
    try:
        files.download(f)
    except Exception as e:
        print('  download manually from the Files panel:', e)

## 9. Per-model report (optional)
Render the single-model HTML report for each model (pure stdlib, no extra
installs). The side-by-side Mistral-vs-Llama comparison is a separate local
Gradio demo that reads these saved JSON files.

In [ ]:
import glob
from IPython.display import HTML, display
for f in sorted(glob.glob('results/pilot_*.json')):
    if f.endswith('pilot_combined.json'): continue
    out = f.replace('.json', '_report.html')
    !python scripts/report.py "{f}" -o "{out}"
    display(HTML(open(out).read()))

## 10. Done — free the GPU
Runtime → Disconnect and delete runtime, so the free GPU is released for your
next session. Send me the two per-model JSON files (and pilot_combined.json)
and I'll build the side-by-side comparison demo.